# Exploratory analysis: the campaign base and the decision it supports

The decision this project supports is **not** "will this client convert?". It is **"the contact team can only call N clients this campaign, so which N do we call?"**

What this notebook has to leave settled before any modelling starts:

1. How rare conversions are, and what a random selection of N clients would return. That number is the floor every later policy has to beat.
2. Which columns are unusable because they are only known after the call.
3. Which client segments exist, how large they are, and how far their conversion rates sit from the base rate.

The formal statement of the decision problem goes in `docs/decisions.md`.

In [ ]:
from pathlib import Path
import sys

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Make src/ importable from the notebook
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT / "src"))

from bank_marketing.config import load_config, resolve  # noqa: E402

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid")

CONFIG = load_config()
CONFIG

In [ ]:
RAW_PATH = resolve(CONFIG["data"]["raw_path"])
SEP = CONFIG["data"]["separator"]
TARGET = CONFIG["data"]["target_column"]
POSITIVE = CONFIG["data"]["positive_label"]

df = pd.read_csv(RAW_PATH, sep=SEP)
print(f"{df.shape[0]:,} clients x {df.shape[1]} columns")
df.head()

In [ ]:
# DuckDB connection over the same file. Two ways in, one dataset:
#   - con.sql("... FROM df ...")            -> queries the pandas DataFrame directly
#   - con.sql("... FROM read_csv_auto(...)") -> queries the file without loading it
con = duckdb.connect()
con.sql("SELECT COUNT(*) AS n_clients FROM df").df()

## 1. The target variable

The whole project rests on the fact that conversions are rare. Before anything else, measure how rare.

Questions to answer here:

- What is the conversion rate, in percentage points and in absolute number of clients?
- If the team can contact `capacity.contacts_available` clients, how many conversions would a *random* selection produce? That number is the floor every later policy must beat.
- What does this imbalance imply for the evaluation metrics used later?

In [ ]:
# Worked example of the query style used throughout. Reuse the pattern below.
con.sql(f"""
    SELECT
        {TARGET}                                   AS outcome,
        COUNT(*)                                   AS n_clients,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
    FROM df
    GROUP BY 1
    ORDER BY n_clients DESC
""").df()

In [ ]:
# TODO: the random-selection baseline.
# How many conversions do we expect if we pick capacity.contacts_available
# clients at random? Write it as a number, every later policy is compared against it.

## 2. Data quality

This dataset has no NULLs, but it does have the string `"unknown"` in several categorical columns, which is a missing value wearing a disguise.

Questions to answer here:

- Which columns contain `"unknown"`, and in what proportion?
- Is `"unknown"` random, or is it concentrated in a segment? If it is concentrated, dropping those rows would quietly remove a group of clients from the campaign.
- `pdays = 999` means "not previously contacted". Sentinel values like this break any model that treats the column as a plain number. How will you encode it?

In [ ]:
# TODO: count "unknown" per column, and look at pdays.

## 3. The leakage trap: `duration`

`duration` is the length of the call in seconds. It is the single strongest predictor in this dataset, and it is **useless for this decision**: the value only exists after the call has already happened. You cannot use it to decide who to call.

It is listed in `config/params.yaml` under `data.leakage_columns` so that the decision is explicit and reviewable.

Questions to answer here:

- How strong is the correlation between `duration` and the target? (Worth measuring once, so you can quote the number in an interview.)
- Are there any other columns that would only be known after the contact?

In [ ]:
LEAKAGE = CONFIG["data"]["leakage_columns"]
# TODO: quantify the leakage, then build the honest feature set.
# features = df.drop(columns=LEAKAGE + [TARGET])

## 4. Segments and heterogeneity

The allocation policy carries a coverage constraint so that no segment is systematically abandoned. Justifying that constraint means knowing what the segments look like now.

Questions to answer here:

- Conversion rate by `job`, `age` band, `education`, `contact` type. Which segments are far from the 11% average?
- How big is each segment? A segment with a high conversion rate but only 200 clients does not move the campaign total.
- Which segments would a naive "top-k by score" policy exclude, just by looking at these base rates?
- The macroeconomic columns (`euribor3m`, `emp.var.rate`, `nr.employed`) move over time, not across clients. What does that mean for a model that has to score *next month's* clients?

In [ ]:
# TODO: conversion rate and volume by segment. Keep both columns side by side,
# the rate alone will mislead you.

## 5. What this notebook has to hand over

`docs/decisions.md` gets filled in from here, in plain language and without the word "model":

| Element | What to state |
| --- | --- |
| Decision variable | What exactly is decided, per client |
| Objective | What is being maximised, in currency |
| Constraints | Capacity, and coverage per segment |
| Value assumptions | Value of a conversion, cost of a contact, and why each number is defensible |
| Known limitations | Non-randomised campaign, so this measures propensity and not the incremental effect of the call |

Three things to be able to state without hesitation before moving to the predictive layer:

1. The decision problem, explained without using the word "model".
2. The value assumption in use, and why it is reasonable.
3. What the imbalance of the target implies for the metrics that come next.